# Training Hub Unified Callbacks — Example Notebook

This notebook validates unified `TrainingHubCallback` support across Training Hub backends on Red Hat OpenShift AI. It submits three TrainJobs, one per backend with `callbacks=` set, and verifies SDK injection and hook output in pod logs.

| # | Backend | Algorithm | `callbacks=` | Expected log behavior |
|---|---------|-----------|--------------|------------------------|
| 1 | Unsloth | `lora_sft` | `UnslothSmokeLogger` | `[UNSLOTH\|CALLBACK\|...]` markers |
| 2 | InstructLab | `sft` | `InstructLabSmokeLogger` | `[ILAB\|CALLBACK\|...]` markers |
| 3 | MiniTrainer | `osft` | `MiniTrainerSmokeLogger` | `[MINI\|CALLBACK\|...]` markers |

**Execution:** Kernel → Restart & Run All (approximately 30 to 60 minutes for all three jobs).

**Prerequisites:** Kubeflow Trainer on cluster, GPU runtime, `training_hub` with unified callbacks.

## 1. Verify SDK

Confirm the workbench SDK exposes `callbacks=` on `TrainingHubTrainer` before submitting jobs. Requires RHOAI 3.6.EA1+ with Kubeflow SDK callbacks support.

In [ ]:
# Connect to the Trainer API and confirm callbacks= is available.
import inspect

from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TrainingHubAlgorithms, TrainingHubTrainer

assert "callbacks" in inspect.signature(TrainingHubTrainer).parameters, (
    "TrainingHubTrainer.callbacks missing. Use RHOAI 3.6.EA1+ with SDK callbacks support"
)

client = TrainerClient()
print("SDK OK — TrainingHubTrainer.callbacks is available")
print("Runtimes:")
for runtime in client.list_runtimes():
    print(f"  - {runtime.name}")

## 2. Configuration

Set the training runtime, model, GPU resources, and Hugging Face cache paths shared by all three TrainJobs. Override values with environment variables when needed.

In [ ]:
import os
from pathlib import Path

# Training runtime and model used by all three jobs.
RUNTIME_NAME = os.environ.get("TRAINING_RUNTIME", "training-hub")

MODEL_PATH = os.environ.get("SMOKE_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
WORKSPACE = Path(os.environ.get("WORKSPACE_DIR", "/opt/app-root/src"))
HF_TOKEN = os.environ.get("HF_TOKEN", "")

CPU = os.environ.get("SMOKE_CPU", "4")
MEMORY = os.environ.get("SMOKE_MEMORY", "16Gi")
GPU = os.environ.get("SMOKE_GPU", "1")

COMMON_ENV = {
    "HF_HOME": str(WORKSPACE / ".cache" / "huggingface"),
    "TOKENIZERS_PARALLELISM": "false",
    **({"HF_TOKEN": HF_TOKEN} if HF_TOKEN else {}),
}
COMMON_RESOURCES = {"cpu": CPU, "memory": MEMORY, "nvidia.com/gpu": GPU}

print(f"runtime={RUNTIME_NAME}")
print(f"model={MODEL_PATH}")

## 3. Write helper modules

The SDK serializes callbacks with `inspect.getsource()`. Write callback and training functions to `.py` files (for example with `%%writefile`) so they can be imported when the TrainJob is submitted.

The next three cells create:
- `hub_callback_smoke.py`: callback classes and log markers per backend
- `smoke_train_func*.py`: small train functions that call each Training Hub backend

In [ ]:
# Callback classes and log markers used to verify hook output per backend.
%%writefile hub_callback_smoke.py
"""Training Hub callbacks with backend-specific log tags for verification."""

from training_hub import TrainingHubCallback, TrainingHubContext


# Each callback class is self-contained. SDK inspect.getsource() serializes leaf
# classes into the train func file without their bases.
class UnslothSmokeLogger(TrainingHubCallback):
    TAG = "UNSLOTH"

    def on_train_begin(self, context: TrainingHubContext) -> None:
        print(
            f"[{self.TAG}|CALLBACK|BEGIN] backend={self.TAG} "
            f"output_dir={context.output_dir} main={context.is_main_process}",
            flush=True,
        )

    def on_log(self, context: TrainingHubContext) -> None:
        print(
            f"[{self.TAG}|CALLBACK|LOG] step={context.step} epoch={context.epoch} "
            f"loss={context.loss} lr={context.learning_rate}",
            flush=True,
        )

    def on_train_end(self, context: TrainingHubContext) -> None:
        print(
            f"[{self.TAG}|CALLBACK|END] step={context.step} backend={self.TAG}",
            flush=True,
        )


class InstructLabSmokeLogger(TrainingHubCallback):
    TAG = "ILAB"

    def on_train_begin(self, context: TrainingHubContext) -> None:
        print(
            f"[{self.TAG}|CALLBACK|BEGIN] backend={self.TAG} "
            f"output_dir={context.output_dir} main={context.is_main_process}",
            flush=True,
        )

    def on_log(self, context: TrainingHubContext) -> None:
        print(
            f"[{self.TAG}|CALLBACK|LOG] step={context.step} epoch={context.epoch} "
            f"loss={context.loss} lr={context.learning_rate}",
            flush=True,
        )

    def on_train_end(self, context: TrainingHubContext) -> None:
        print(
            f"[{self.TAG}|CALLBACK|END] step={context.step} backend={self.TAG}",
            flush=True,
        )


class MiniTrainerSmokeLogger(TrainingHubCallback):
    TAG = "MINI"

    def on_train_begin(self, context: TrainingHubContext) -> None:
        print(
            f"[{self.TAG}|CALLBACK|BEGIN] backend={self.TAG} "
            f"output_dir={context.output_dir} main={context.is_main_process}",
            flush=True,
        )

    def on_log(self, context: TrainingHubContext) -> None:
        print(
            f"[{self.TAG}|CALLBACK|LOG] step={context.step} epoch={context.epoch} "
            f"loss={context.loss} lr={context.learning_rate}",
            flush=True,
        )

    def on_train_end(self, context: TrainingHubContext) -> None:
        print(
            f"[{self.TAG}|CALLBACK|END] step={context.step} backend={self.TAG}",
            flush=True,
        )


UNSLOTH_CALLBACK_MARKERS = ["[UNSLOTH|CALLBACK|BEGIN]", "[UNSLOTH|CALLBACK|LOG]", "[UNSLOTH|CALLBACK|END]"]
ILAB_CALLBACK_MARKERS = ["[ILAB|CALLBACK|BEGIN]", "[ILAB|CALLBACK|LOG]", "[ILAB|CALLBACK|END]"]
MINI_CALLBACK_MARKERS = ["[MINI|CALLBACK|BEGIN]", "[MINI|CALLBACK|LOG]", "[MINI|CALLBACK|END]"]


In [ ]:
# Train function for the Unsloth lora_sft backend.
%%writefile smoke_train_func.py
"""Unsloth backend — calls training_hub.lora_sft()."""


def callback_smoke_train(**kwargs) -> None:
    import json
    from pathlib import Path

    from training_hub import lora_sft

    data_path = Path("/tmp/callback_smoke_data.jsonl")
    rows = [
        {"messages": [{"role": "user", "content": "What is 2+2?"}, {"role": "assistant", "content": "4"}]},
        {"messages": [{"role": "user", "content": "Capital of France?"}, {"role": "assistant", "content": "Paris"}]},
        {"messages": [{"role": "user", "content": "Say hello"}, {"role": "assistant", "content": "Hello!"}]},
        {"messages": [{"role": "user", "content": "Color of the sky?"}, {"role": "assistant", "content": "Blue"}]},
    ]
    with data_path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")

    args = dict(kwargs)
    args["data_path"] = str(data_path)
    args.setdefault("ckpt_output_dir", "/tmp/callback_smoke_out")
    print(f"[PY] Launching lora_sft (Unsloth) data={data_path}", flush=True)
    lora_sft(**args)
    print("[PY] lora_sft complete", flush=True)

In [ ]:
# Train function for the InstructLab sft backend.
%%writefile smoke_train_func_sft.py
"""InstructLab backend — calls training_hub.sft()."""


def callback_smoke_train_sft(**kwargs) -> None:
    import json
    from pathlib import Path

    from training_hub import sft

    data_path = Path("/tmp/callback_smoke_data_sft.jsonl")
    rows = [
        {"messages": [{"role": "user", "content": "What is 2+2?"}, {"role": "assistant", "content": "4"}]},
        {"messages": [{"role": "user", "content": "Capital of France?"}, {"role": "assistant", "content": "Paris"}]},
        {"messages": [{"role": "user", "content": "Say hello"}, {"role": "assistant", "content": "Hello!"}]},
        {"messages": [{"role": "user", "content": "Color of the sky?"}, {"role": "assistant", "content": "Blue"}]},
    ]
    with data_path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")

    args = dict(kwargs)
    args["data_path"] = str(data_path)
    args.setdefault("ckpt_output_dir", "/tmp/callback_smoke_out_sft")
    args.setdefault("max_tokens_per_gpu", 4096)
    args.setdefault("mlflow_tracking_uri", "")
    print(f"[PY] Launching sft (InstructLab) data={data_path}", flush=True)
    sft(**args)
    print("[PY] sft complete", flush=True)

In [ ]:
# Train function for the MiniTrainer osft backend.
%%writefile smoke_train_func_osft.py
"""MiniTrainer backend — calls training_hub.osft()."""


def callback_smoke_train_osft(**kwargs) -> None:
    import json
    from pathlib import Path

    from training_hub import osft

    data_path = Path("/tmp/callback_smoke_data_osft.jsonl")
    rows = [
        {"messages": [{"role": "user", "content": "What is 2+2?"}, {"role": "assistant", "content": "4"}]},
        {"messages": [{"role": "user", "content": "Capital of France?"}, {"role": "assistant", "content": "Paris"}]},
        {"messages": [{"role": "user", "content": "Say hello"}, {"role": "assistant", "content": "Hello!"}]},
        {"messages": [{"role": "user", "content": "Color of the sky?"}, {"role": "assistant", "content": "Blue"}]},
    ]
    with data_path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row) + "\n")

    args = dict(kwargs)
    args["data_path"] = str(data_path)
    args.setdefault("ckpt_output_dir", "/tmp/callback_smoke_out_osft")
    args.setdefault("unfreeze_rank_ratio", 0.1)
    args.setdefault("max_tokens_per_gpu", 4096)
    args.setdefault("mlflow_tracking_uri", "")
    print(f"[PY] Launching osft (MiniTrainer) data={data_path}", flush=True)
    osft(**args)
    print("[PY] osft complete", flush=True)

## 4. Shared helpers

Define helpers to submit a TrainJob, wait for completion, fetch pod logs, and check for SDK injection plus backend callback markers.

In [ ]:
# Helpers to submit jobs, fetch logs, and record pass or fail results.
from kubeflow.trainer.constants import constants
from kubeflow.trainer.rhai import TrainingHubTrainer

SDK_MARKERS = [
    "Prepared 1 Training Hub callback",
    "Training Hub callback injection configured",
]

VERIFICATION_RESULTS: dict[str, bool] = {}


def wait_and_get_logs(job_name: str, timeout: int = 3600) -> tuple[bool, str]:
    """Wait for TrainJob completion and return (completed, log_text)."""
    completed = False
    try:
        job = client.wait_for_job_status(
            name=job_name,
            status={constants.TRAINJOB_COMPLETE},
            timeout=timeout,
            polling_interval=10,
        )
        print(f"TrainJob {job_name} status: {job.status}")
        completed = True
    except Exception as exc:
        print(f"TrainJob {job_name} did not complete: {exc}")
    try:
        logs = "".join(client.get_job_logs(name=job_name, follow=False, step="node-0"))
    except Exception as exc:
        print(f"Could not fetch logs for {job_name}: {exc}")
        logs = ""
    return completed, logs


def verify_logs(
    logs: str,
    *,
    label: str,
    expect_callbacks: bool,
    result_key: str,
    callback_markers: list[str],
    job_completed: bool,
) -> bool:
    """Verify pod logs contain expected SDK and callback markers."""
    has_sdk = all(marker in logs for marker in SDK_MARKERS)
    has_hooks = all(marker in logs for marker in callback_markers)

    print(f"\n{'=' * 60}")
    print(label)
    print(f"{'=' * 60}")
    print(f"Job completed: {job_completed}")
    print(f"Expect callbacks: {expect_callbacks}")

    print("\nSDK injection markers:")
    for marker in SDK_MARKERS:
        found = marker in logs
        print(f"  {'PASS' if found else 'absent'}  {marker}")

    print("\nBackend callback markers:")
    for marker in callback_markers:
        found = marker in logs
        print(f"  {'PASS' if found else 'absent'}  {marker}")

    if not job_completed:
        passed = False
        print("\nRESULT: ❌ FAILED: job did not complete successfully")
    elif expect_callbacks:
        passed = has_sdk and has_hooks
        print(
            f"\nRESULT: {'✅ WITH CALLBACKS OK' if passed else '❌ WITH CALLBACKS FAILED'}"
        )
    else:
        passed = not any(m in logs for m in SDK_MARKERS) and not any(
            m in logs for m in callback_markers
        )
        print(
            f"\nRESULT: {'✅ BASELINE OK (no callbacks)' if passed else '❌ BASELINE UNEXPECTED MARKERS'}"
        )

    VERIFICATION_RESULTS[result_key] = passed
    return passed


def submit_and_verify(
    trainer: TrainingHubTrainer,
    *,
    label: str,
    expect_callbacks: bool,
    result_key: str,
    callback_markers: list[str],
) -> str:
    """Submit TrainJob, wait, verify logs. Returns job name."""
    job_name = client.train(runtime=RUNTIME_NAME, trainer=trainer)
    print(f"Submitted: {job_name}")
    completed, logs = wait_and_get_logs(job_name)
    print("\n--- pod logs (tail) ---")
    tail = logs[-4000:] if len(logs) > 4000 else logs
    print(tail)
    verify_logs(
        logs,
        label=label,
        expect_callbacks=expect_callbacks,
        result_key=result_key,
        callback_markers=callback_markers,
        job_completed=completed,
    )
    return job_name

---
# Part A — Unsloth (`lora_sft`)

Submit a LoRA SFT job through the Unsloth backend with `callbacks=[UnslothSmokeLogger]`. Pod logs should include `[UNSLOTH|CALLBACK|...]` markers.

### A · Unsloth with unified callback

Build the Unsloth trainer with `callbacks=` and submit the job.

In [ ]:
# Submit the TrainJob and verify callback markers in pod logs.
from hub_callback_smoke import UNSLOTH_CALLBACK_MARKERS, UnslothSmokeLogger
from smoke_train_func import callback_smoke_train

trainer_unsloth_after = TrainingHubTrainer(
    func=callback_smoke_train,
    algorithm=TrainingHubAlgorithms.LORA_SFT,
    func_args={
        "model_path": MODEL_PATH,
        "ckpt_output_dir": "/tmp/callback_smoke_out_after",
        "num_epochs": 1,
        "max_seq_len": 128,
        "micro_batch_size": 1,
        "logging_steps": 1,
        "save_steps": 9999,
        "save_total_limit": 1,
        "warmup_steps": 0,
        "learning_rate": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "sample_packing": True,
    },
    callbacks=[UnslothSmokeLogger],
    resources_per_node=COMMON_RESOURCES,
    enable_progression_tracking=False,
    env=COMMON_ENV,
)

unsloth_after_job = submit_and_verify(
    trainer_unsloth_after,
    label="A · Unsloth (lora_sft) — UnslothSmokeLogger",
    expect_callbacks=True,
    result_key="unsloth_after",
    callback_markers=UNSLOTH_CALLBACK_MARKERS,
)

---
# Part B — InstructLab Training (`sft`)

Submit an InstructLab SFT job with `callbacks=[InstructLabSmokeLogger]`. Pod logs should include `[ILAB|CALLBACK|...]` markers.

### B · InstructLab with unified callback

Build the InstructLab trainer with `callbacks=` and submit the job.

In [ ]:
# Submit the TrainJob and verify callback markers in pod logs.
from hub_callback_smoke import ILAB_CALLBACK_MARKERS, InstructLabSmokeLogger
from smoke_train_func_sft import callback_smoke_train_sft

trainer_sft_after = TrainingHubTrainer(
    func=callback_smoke_train_sft,
    algorithm=TrainingHubAlgorithms.SFT,
    func_args={
        "model_path": MODEL_PATH,
        "ckpt_output_dir": "/tmp/callback_smoke_out_sft_after",
        "num_epochs": 1,
        "max_seq_len": 128,
        "effective_batch_size": 4,
        "learning_rate": 2e-5,
        "max_tokens_per_gpu": 4096,
        "mlflow_tracking_uri": "",
    },
    callbacks=[InstructLabSmokeLogger],
    resources_per_node=COMMON_RESOURCES,
    enable_progression_tracking=False,
    env=COMMON_ENV,
)

sft_after_job = submit_and_verify(
    trainer_sft_after,
    label="B · InstructLab (sft) — InstructLabSmokeLogger",
    expect_callbacks=True,
    result_key="sft_after",
    callback_markers=ILAB_CALLBACK_MARKERS,
)

---
# Part C — MiniTrainer (`osft`)

Submit a MiniTrainer OSFT job with `callbacks=[MiniTrainerSmokeLogger]`. Pod logs should include `[MINI|CALLBACK|...]` markers.

### C · MiniTrainer with unified callback

Build the MiniTrainer trainer with `callbacks=` and submit the job.

In [ ]:
# Submit the TrainJob and verify callback markers in pod logs.
from hub_callback_smoke import MINI_CALLBACK_MARKERS, MiniTrainerSmokeLogger
from smoke_train_func_osft import callback_smoke_train_osft

trainer_osft_after = TrainingHubTrainer(
    func=callback_smoke_train_osft,
    algorithm=TrainingHubAlgorithms.OSFT,
    func_args={
        "model_path": MODEL_PATH,
        "ckpt_output_dir": "/tmp/callback_smoke_out_osft_after",
        "num_epochs": 1,
        "max_seq_len": 128,
        "effective_batch_size": 4,
        "learning_rate": 2e-5,
        "unfreeze_rank_ratio": 0.1,
        "max_tokens_per_gpu": 4096,
        "mlflow_tracking_uri": "",
    },
    callbacks=[MiniTrainerSmokeLogger],
    resources_per_node=COMMON_RESOURCES,
    enable_progression_tracking=False,
    env=COMMON_ENV,
)

osft_after_job = submit_and_verify(
    trainer_osft_after,
    label="C · MiniTrainer (osft) — MiniTrainerSmokeLogger",
    expect_callbacks=True,
    result_key="osft_after",
    callback_markers=MINI_CALLBACK_MARKERS,
)

---
# Verification summary

Print a pass or fail table for all three backend runs. The notebook raises if any scenario fails.

In [ ]:
# Print the final pass or fail table for all three backends.
ROWS = [
    ("Unsloth", "lora_sft", "UnslothSmokeLogger", "unsloth_after"),
    ("InstructLab", "sft", "InstructLabSmokeLogger", "sft_after"),
    ("MiniTrainer", "osft", "MiniTrainerSmokeLogger", "osft_after"),
]

print("\n" + "=" * 72)
print("Verification summary: Unified Training Hub Callbacks")
print("=" * 72)
print(f"{'Backend':<14} {'Algo':<10} {'Callback':<35} {'Result':<8}")
print("-" * 72)

all_pass = True
for backend, algo, scenario, key in ROWS:
    if key not in VERIFICATION_RESULTS:
        status = "SKIP"
        all_pass = False
    else:
        status = "PASS" if VERIFICATION_RESULTS[key] else "FAIL"
        all_pass = all_pass and VERIFICATION_RESULTS[key]
    print(f"{backend:<14} {algo:<10} {scenario:<35} {status:<8}")

print("-" * 72)
if all_pass and len(VERIFICATION_RESULTS) == len(ROWS):
    print("\nAll three scenarios passed.")
    print("Callback markers verified per backend: UNSLOTH | ILAB | MINI.")
else:
    print("\nOne or more scenarios failed. Review the rows marked FAIL or SKIP above.")
    raise RuntimeError("Callback smoke verification failed — see summary table above.")

## Troubleshooting

| Symptom | Fix |
|---------|-----|
| `callbacks` missing on `TrainingHubTrainer` | Use RHOAI 3.6.EA1+ with SDK callbacks support |
| `inspect.getsource` error | Define callback in `hub_callback_smoke.py` via `%%writefile`, not inline in a cell |
| `NameError` for callback base class | Define all hooks on the concrete class passed to `callbacks=` |
| SFT/OSFT MLflow error | `mlflow_tracking_uri=""` already set in func_args |
| 403 on runtimes | Grant notebook SA RBAC for TrainJobs |